In [3]:
from IPython.display import HTML, display

APP = r'''<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>OS Scheduling Visualizer</title>
<style>
:root{box-sizing:border-box;padding-top:env(safe-area-inset-top,0px);padding-bottom:env(safe-area-inset-bottom,0px);--bg:#f6f7f9;--card:#fff;--tx:#1c2430;--mu:#5c6676;--bd:#d9dee6;--ac:#2563eb;--ok:#16a34a;--wr:#dc2626}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){color-scheme:dark;--bg:#12161c;--card:#1b212a;--tx:#e6eaf0;--mu:#9aa5b5;--bd:#2e3745;--ac:#60a5fa;--ok:#4ade80;--wr:#f87171}}
:root[data-theme="dark"]{color-scheme:dark;--bg:#12161c;--card:#1b212a;--tx:#e6eaf0;--mu:#9aa5b5;--bd:#2e3745;--ac:#60a5fa;--ok:#4ade80;--wr:#f87171}
html{scroll-padding-top:env(safe-area-inset-top,0px)}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--tx);font:15px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
main{max-width:980px;margin:0 auto;padding:16px}
h1{font-size:22px;margin:4px 0}h2{font-size:16px;margin:0 0 8px}
.sub{color:var(--mu);margin:0 0 12px}
.tabs{display:flex;gap:8px;margin:12px 0}
button,select,input,textarea{font:inherit;color:var(--tx);background:var(--card);border:1px solid var(--bd);border-radius:8px;padding:7px 10px}
button{cursor:pointer}button:hover{border-color:var(--ac)}
button.on,button.pri{background:var(--ac);border-color:var(--ac);color:#fff}
.card{background:var(--card);border:1px solid var(--bd);border-radius:12px;padding:14px;margin-bottom:14px}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:10px}
label{display:block;font-size:13px;color:var(--mu)}label input,label select,label textarea{width:100%;margin-top:3px}
textarea{font-family:ui-monospace,Menlo,monospace;min-height:110px}
.row{display:flex;flex-wrap:wrap;gap:8px;align-items:center;margin-top:10px}
.err{color:var(--wr);font-size:13px;min-height:18px;margin-top:6px}
.stat{display:flex;flex-wrap:wrap;gap:18px;margin:8px 0}.stat b{font-size:20px;display:block}
.scroll{overflow-x:auto}
svg text{fill:var(--mu);font-size:11px}
table{border-collapse:collapse;width:100%;font-size:14px}th,td{border-bottom:1px solid var(--bd);padding:6px 8px;text-align:right}th:first-child,td:first-child{text-align:left}
.seq span{display:inline-block;padding:1px 8px;margin:2px;border-radius:999px;border:1px solid var(--bd)}
.seq span.cur{background:var(--ac);color:#fff;border-color:var(--ac)}
.gantt{display:flex;height:44px;min-width:100%;margin-top:6px}
.gantt div{display:flex;align-items:center;justify-content:center;color:#fff;font-size:12px;border-right:1px solid var(--card);min-width:0;overflow:hidden}
.ticks{display:flex;min-width:100%;font-size:11px;color:var(--mu)}.ticks div{border-left:1px solid var(--bd);padding-left:2px;overflow:hidden}
.hide{display:none}
</style></head><body><main>
<h1>OS Scheduling Visualizer</h1>
<p class="sub">An open learning aid for disk and CPU scheduling. Runs offline in any browser.</p>
<div class="tabs"><button id="tD" class="on">Disk scheduling</button><button id="tC">CPU scheduling</button></div>

<section id="disk">
<div class="card"><h2>Input</h2><div class="grid">
<label>Disk size (cylinders)<input id="size" type="number" value="200"></label>
<label>Initial head<input id="head" type="number" value="50"></label>
<label>Direction (SCAN / LOOK / C-SCAN)<select id="dir"><option value="up">Towards higher</option><option value="down">Towards lower</option></select></label>
<label>Algorithm<select id="alg"><option>FCFS</option><option>SSTF</option><option>SCAN</option><option>C-SCAN</option><option>LOOK</option></select></label>
</div>
<label style="margin-top:10px">Request queue (comma separated)<input id="reqs" value="82, 170, 43, 140, 24, 16, 190"></label>
<div class="err" id="derr"></div>
<div class="row"><button class="pri" id="dRun">Run</button><button id="dPrev">◀ Previous</button><button id="dNext">Next ▶</button><button id="dPlay">▶ Auto-play</button><button id="dReset">Reset</button>
<label style="display:inline-flex;gap:6px;align-items:center">Speed<input id="dSpd" type="range" min="1" max="10" value="5" style="width:110px"></label></div></div>
<div class="card"><h2>Seek graph</h2><div class="scroll"><svg id="seek" width="100%" viewBox="0 0 900 360"></svg></div>
<div class="stat"><div>Total head movement<b id="tot">–</b></div><div>Movement so far<b id="sofar">–</b></div></div>
<div>Service sequence: <span class="seq" id="seq"></span></div>
<p class="sub" style="margin-top:8px">SCAN and C-SCAN travel to the disk end. C-SCAN counts the return jump. LOOK and SSTF stop at the last request.</p></div>
<div class="card"><h2>Comparison: total head movement</h2><div class="scroll"><svg id="bars" width="100%" viewBox="0 0 900 260"></svg></div></div>
</section>

<section id="cpu" class="hide">
<div class="card"><h2>Input</h2>
<label>Processes, one per line: ID, arrival, burst, priority (lower number = higher priority)<textarea id="procs">P1,0,8,3
P2,1,4,1
P3,2,9,4
P4,3,5,2</textarea></label>
<div class="grid" style="margin-top:10px"><label>Algorithm<select id="calg"><option>FCFS</option><option>SJF</option><option>SRTF</option><option>Priority</option><option>Round Robin</option></select></label>
<label>Time quantum (Round Robin)<input id="tq" type="number" value="3"></label></div>
<div class="err" id="cerr"></div>
<div class="row"><button class="pri" id="cRun">Run</button><button id="cPrev">◀ Previous</button><button id="cNext">Next ▶</button><button id="cPlay">▶ Auto-play</button><button id="cReset">Reset</button></div></div>
<div class="card"><h2>Gantt chart</h2><div class="scroll"><div id="gantt"></div><div id="ticks"></div></div></div>
<div class="card"><h2>Metrics</h2><div class="scroll"><table id="met"></table></div></div>
</section>
</main>
<script>
const $=id=>document.getElementById(id);
const NS="http://www.w3.org/2000/svg";
const col=i=>`hsl(${(i*67+210)%360} 60% 45%)`;
/* ---------- tabs ---------- */
$('tD').onclick=()=>tab(true);$('tC').onclick=()=>tab(false);
function tab(d){$('disk').classList.toggle('hide',!d);$('cpu').classList.toggle('hide',d);$('tD').classList.toggle('on',d);$('tC').classList.toggle('on',!d)}
/* ---------- disk algorithms ---------- */
function diskPath(alg,reqs,head,size,dir){
  const up=dir==='up',last=size-1;
  const lo=reqs.filter(r=>r<head).sort((a,b)=>b-a);   // below head, nearest first
  const hi=reqs.filter(r=>r>=head).sort((a,b)=>a-b);  // at/above head, nearest first
  let p=[head];
  if(alg==='FCFS')p.push(...reqs);
  else if(alg==='SSTF'){let c=head,rest=reqs.slice();while(rest.length){let bi=0;rest.forEach((r,i)=>{if(Math.abs(r-c)<Math.abs(rest[bi]-c))bi=i});c=rest.splice(bi,1)[0];p.push(c)}}
  else if(alg==='SCAN'){if(up){p.push(...hi,last,...lo)}else{p.push(...lo,0,...hi)}}
  else if(alg==='C-SCAN'){if(up){p.push(...hi,last);if(lo.length)p.push(0,...lo.slice().reverse())}else{p.push(...lo,0);if(hi.length)p.push(last,...hi.slice().reverse())}}
  else if(alg==='LOOK'){if(up)p.push(...hi,...lo);else p.push(...lo,...hi)}
  const t=p.slice(1).reduce((s,v,i)=>s+Math.abs(v-p[i]),0);
  return{path:p,total:t};
}
let D=null,dk=0,dTimer=null;
function readDisk(){
  const size=+$('size').value,head=+$('head').value;
  if(!Number.isInteger(size)||size<2)return'Disk size must be an integer of at least 2.';
  if(!Number.isInteger(head)||head<0||head>=size)return`Head position must be an integer between 0 and ${size-1}.`;
  const raw=$('reqs').value.split(/[\s,]+/).filter(Boolean);
  if(!raw.length)return'Enter at least one request.';
  const reqs=raw.map(Number);
  if(reqs.some(r=>!Number.isInteger(r)))return'Requests must be whole numbers separated by commas.';
  const bad=reqs.find(r=>r<0||r>=size);
  if(bad!==undefined)return`Request ${bad} is outside the disk range 0 to ${size-1}.`;
  return{size,head,reqs,dir:$('dir').value};
}
function dRun(){
  stop();const r=readDisk();
  if(typeof r==='string'){$('derr').textContent=r;return}
  $('derr').textContent='';
  D={...r,alg:$('alg').value};D.res=diskPath(D.alg,D.reqs,D.head,D.size,D.dir);dk=0;drawDisk();drawBars();
}
function drawDisk(){
  const svg=$('seek');svg.innerHTML='';if(!D)return;
  const {path}=D.res,W=900,H=360,m={l:20,r:20,t:30,b:10},n=path.length-1;
  const X=c=>m.l+c/(D.size-1)*(W-m.l-m.r),Y=i=>m.t+i/Math.max(n,1)*(H-m.t-m.b);
  const mk=(t,a,txt)=>{const e=document.createElementNS(NS,t);for(const k in a)e.setAttribute(k,a[k]);if(txt!==undefined)e.textContent=txt;svg.appendChild(e);return e};
  mk('line',{x1:m.l,x2:W-m.r,y1:m.t-8,y2:m.t-8,stroke:'var(--bd)'});
  [0,Math.round((D.size-1)/2),D.size-1].forEach(c=>mk('text',{x:X(c),y:12,'text-anchor':'middle'},c));
  const set=new Set(D.reqs);
  path.forEach((c,i)=>{ if(i<=dk){ if(i>0)mk('line',{x1:X(path[i-1]),y1:Y(i-1),x2:X(c),y2:Y(i),stroke:'var(--ac)','stroke-width':2});
    const isReq=i>0&&set.has(c)&&D.reqs.includes(c);
    mk('circle',{cx:X(c),cy:Y(i),r:i===dk?6:4,fill:i===0?'var(--ok)':(i===dk?'var(--wr)':'var(--ac)')});
    mk('text',{x:X(c)+8,y:Y(i)+4},c+(i===0?' (start)':''));}
  });
  const sofar=path.slice(0,dk+1).reduce((s,v,i)=>i?s+Math.abs(v-path[i-1]):0,0);
  $('tot').textContent=D.res.total;$('sofar').textContent=sofar;
  const served=[];let sset=D.reqs.slice();
  for(let i=1;i<=dk;i++){const ix=sset.indexOf(path[i]);if(ix>=0){served.push({c:path[i],cur:i===dk});sset.splice(ix,1)}}
  $('seq').innerHTML=served.map(s=>`<span class="${s.cur?'cur':''}">${s.c}</span>`).join('')||'<span>–</span>';
}
function drawBars(){
  const svg=$('bars');svg.innerHTML='';const algs=['FCFS','SSTF','SCAN','C-SCAN','LOOK'];
  const vals=algs.map(a=>diskPath(a,D.reqs,D.head,D.size,D.dir).total),mx=Math.max(...vals),best=Math.min(...vals);
  const bw=100,gap=60,base=220;
  algs.forEach((a,i)=>{const h=vals[i]/mx*170,x=60+i*(bw+gap);
    const r=document.createElementNS(NS,'rect');r.setAttribute('x',x);r.setAttribute('y',base-h);r.setAttribute('width',bw);r.setAttribute('height',h);r.setAttribute('rx',6);
    r.setAttribute('fill',vals[i]===best?'var(--ok)':(a===D.alg?'var(--ac)':'var(--mu)'));r.setAttribute('opacity',a===D.alg||vals[i]===best?1:.55);svg.appendChild(r);
    [[a,base+16],[vals[i],base-h-6]].forEach(([t,y])=>{const e=document.createElementNS(NS,'text');e.setAttribute('x',x+bw/2);e.setAttribute('y',y);e.setAttribute('text-anchor','middle');e.style.fontSize='13px';e.textContent=t;svg.appendChild(e)});
  });
}
function stop(){clearInterval(dTimer);clearInterval(cTimer);dTimer=cTimer=null;$('dPlay').textContent='▶ Auto-play';$('cPlay').textContent='▶ Auto-play'}
$('dRun').onclick=dRun;
$('dNext').onclick=()=>{if(!D)dRun();if(D&&dk<D.res.path.length-1){dk++;drawDisk()}};
$('dPrev').onclick=()=>{if(D&&dk>0){dk--;drawDisk()}};
$('dReset').onclick=()=>{stop();if(D){dk=0;drawDisk()}};
$('dPlay').onclick=()=>{ if(dTimer){stop();return} if(!D)dRun(); if(!D)return; if(dk>=D.res.path.length-1)dk=0;
  $('dPlay').textContent='⏸ Pause';dTimer=setInterval(()=>{if(dk>=D.res.path.length-1){stop();return}dk++;drawDisk()},1100-$('dSpd').value*100)};
/* ---------- CPU algorithms ---------- */
function readProcs(){
  const lines=$('procs').value.split('\n').map(l=>l.trim()).filter(Boolean);
  if(!lines.length)return'Enter at least one process.';
  const ps=[],ids=new Set();
  for(const [i,l] of lines.entries()){
    const f=l.split(/[\s,]+/);
    if(f.length<4)return`Line ${i+1}: expected ID, arrival, burst, priority.`;
    const [id,a,b,p]=[f[0],+f[1],+f[2],+f[3]];
    if(ids.has(id))return`Duplicate process ID ${id}.`;ids.add(id);
    if(!(a>=0)||!Number.isInteger(a))return`Line ${i+1}: arrival must be a whole number ≥ 0.`;
    if(!(b>=1)||!Number.isInteger(b))return`Line ${i+1}: burst must be a whole number ≥ 1.`;
    if(!Number.isInteger(p))return`Line ${i+1}: priority must be a whole number.`;
    ps.push({id,at:a,bt:b,pr:p,i});
  }
  return ps;
}
function cpu(alg,ps,q){
  const seg=[],rem={},ct={};ps.forEach(p=>rem[p.id]=p.bt);
  const push=(id,s,e)=>{const l=seg[seg.length-1];if(l&&l.id===id&&l.e===s)l.e=e;else seg.push({id,s,e})};
  let t=0,done=0;const n=ps.length;
  if(alg==='Round Robin'){
    const sorted=ps.slice().sort((a,b)=>a.at-b.at||a.i-b.i);let nx=0,queue=[];
    const arr=()=>{while(nx<n&&sorted[nx].at<=t)queue.push(sorted[nx++])};
    while(done<n){arr();
      if(!queue.length){const na=sorted[nx].at;seg.push({id:null,s:t,e:na});t=na;continue}
      const p=queue.shift(),r=Math.min(q,rem[p.id]);seg.push({id:p.id,s:t,e:t+r});t+=r;rem[p.id]-=r;arr();
      if(rem[p.id]>0)queue.push(p);else{ct[p.id]=t;done++}}
  } else if(alg==='SRTF'){
    while(done<n){const rd=ps.filter(p=>p.at<=t&&rem[p.id]>0);
      if(!rd.length){const na=Math.min(...ps.filter(p=>rem[p.id]>0).map(p=>p.at));push(null,t,na);t=na;continue}
      rd.sort((a,b)=>rem[a.id]-rem[b.id]||a.at-b.at||a.i-b.i);const p=rd[0];
      push(p.id,t,t+1);t++;rem[p.id]--;if(!rem[p.id]){ct[p.id]=t;done++}}
  } else {
    const key={FCFS:p=>[p.at,p.i],SJF:p=>[p.bt,p.at,p.i],Priority:p=>[p.pr,p.at,p.i]}[alg];
    while(done<n){const rd=ps.filter(p=>p.at<=t&&rem[p.id]>0);
      if(!rd.length){const na=Math.min(...ps.filter(p=>rem[p.id]>0).map(p=>p.at));seg.push({id:null,s:t,e:na});t=na;continue}
      rd.sort((a,b)=>{const x=key(a),y=key(b);for(let k=0;k<x.length;k++)if(x[k]!==y[k])return x[k]-y[k];return 0});
      const p=rd[0];seg.push({id:p.id,s:t,e:t+p.bt});t+=p.bt;rem[p.id]=0;ct[p.id]=t;done++}
  }
  const rows=ps.map(p=>({id:p.id,at:p.at,bt:p.bt,pr:p.pr,ct:ct[p.id],tat:ct[p.id]-p.at,wt:ct[p.id]-p.at-p.bt}));
  return{seg,rows};
}
let C=null,ck=0,cTimer=null;
function cRun(){
  stop();const ps=readProcs();
  if(typeof ps==='string'){$('cerr').textContent=ps;return}
  const q=+$('tq').value,alg=$('calg').value;
  if(alg==='Round Robin'&&(!Number.isInteger(q)||q<1)){$('cerr').textContent='Time quantum must be a whole number ≥ 1.';return}
  $('cerr').textContent='';C={ps,alg,...cpu(alg,ps,q)};ck=0;drawCpu();
}
function drawCpu(){
  if(!C)return;const total=C.seg[C.seg.length-1].e,sc=Math.max(28,Math.min(60,860/total));
  const w=total*sc;let g='<div class="gantt" style="width:'+w+'px">';
  C.seg.forEach((s,i)=>{const on=i<ck;const idx=C.ps.findIndex(p=>p.id===s.id);
    g+=`<div style="width:${(s.e-s.s)*sc}px;background:${s.id===null?'var(--mu)':col(idx)};opacity:${on?1:.12}">${on?(s.id||'idle'):''}</div>`});
  $('gantt').innerHTML=g+'</div>';
  let t='<div class="ticks" style="width:'+w+'px">';C.seg.forEach((s,i)=>{t+=`<div style="width:${(s.e-s.s)*sc}px">${i<ck||i===0?s.s:''}${i===ck-1?'<span style="float:right;padding-right:2px">'+s.e+'</span>':''}</div>`});
  $('ticks').innerHTML=t+'</div>';
  const R=C.rows,avg=k=>(R.reduce((s,r)=>s+r[k],0)/R.length).toFixed(2);
  $('met').innerHTML='<tr><th>Process</th><th>Arrival</th><th>Burst</th><th>Priority</th><th>Completion</th><th>Turnaround</th><th>Waiting</th></tr>'+
   R.map(r=>`<tr><td>${r.id}</td><td>${r.at}</td><td>${r.bt}</td><td>${r.pr}</td><td>${r.ct}</td><td>${r.tat}</td><td>${r.wt}</td></tr>`).join('')+
   `<tr><td><b>Average</b></td><td></td><td></td><td></td><td></td><td><b>${avg('tat')}</b></td><td><b>${avg('wt')}</b></td></tr>`;
}
$('cRun').onclick=cRun;
$('cNext').onclick=()=>{if(!C)cRun();if(C&&ck<C.seg.length){ck++;drawCpu()}};
$('cPrev').onclick=()=>{if(C&&ck>0){ck--;drawCpu()}};
$('cReset').onclick=()=>{stop();if(C){ck=0;drawCpu()}};
$('cPlay').onclick=()=>{ if(cTimer){stop();return} if(!C)cRun(); if(!C)return; if(ck>=C.seg.length)ck=0;
  $('cPlay').textContent='⏸ Pause';cTimer=setInterval(()=>{if(ck>=C.seg.length){stop();return}ck++;drawCpu()},800)};
dRun();cRun();
</script></body></html>
'''

display(HTML(APP))
